In [7]:
import pandas as pd 
path = r"C:\Users\nadia\OneDrive\Desktop\Leticia\Documentos\Projetos\cliente_confeitaria\Analise_financeira_confeitaria"
df = pd.read_csv(path + r"\Dados\tratados\extratos_pagbank.csv", sep=";",decimal=",",parse_dates=["Data"], dayfirst=True)

In [8]:
df.sort_values(by="id_linha", inplace=True)
df.dtypes

arquivo_origem               str
Data              datetime64[us]
Descrição                    str
Valor                    float64
Saldo                    float64
id_linha                   int64
dtype: object

Conferencia de saldo mês a mês

In [ ]:
base=df.sort_values(by=["Data", "id_linha"], inplace=False).reset_index(drop=True) #coloca tudo em ordem de data
base["mês"]=base["Data"].dt.to_period("M") #cria coluna com o mês e ano
base[["Data","mês"]].head() #conferência da coluna criada

,Data,mês
0,2025-01-01,2025-01
1,2025-01-02,2025-01
2,2025-01-02,2025-01
3,2025-01-03,2025-01
4,2025-01-03,2025-01


In [ ]:
resumo = base.groupby("mês").agg(
    saldo_1a_linha=("Saldo", "first"),
    valor_1a_linha=("Valor", "first"),
    saldo_final=("Saldo", "last"),
    entradas=("Valor", lambda v: v[v > 0].sum()),
    saidas=("Valor", lambda v: v[v < 0].sum()),
) #Agrupamento mes a mes, e calculo do saldo inicial, saldo final, entradas e saidas para cada grupo

resumo["saldo_inicial"] = resumo["saldo_1a_linha"] - resumo["valor_1a_linha"]
resumo["saldo_calculado"] = resumo["saldo_inicial"] + resumo["entradas"] + resumo["saidas"]
resumo["dif_no_mes"] = (resumo["saldo_calculado"] - resumo["saldo_final"]).round(2)

print(resumo[["saldo_inicial", "entradas", "saidas", "saldo_final", "dif_no_mes"]].to_string())
#O output mostra que todos os dados da base estão corretos, pois comparando com os extratos bancários o saldo incial e final mês a mês bate com o calculado,
#e a diferença no mês é 0,00 para todos os meses.Sendo assim,não é necessáraio uma checagem linha a linha.



         saldo_inicial  entradas    saidas  saldo_final  dif_no_mes
mês                                                                
2025-01       19840.55  22528.48 -31228.40     11140.63         0.0
2025-02       11140.63  70785.00 -31709.43     50216.20         0.0
2025-03       50216.20  36689.52 -76063.54     10842.18        -0.0
2025-04       10842.18  48582.14 -39298.76     20125.56        -0.0
2025-05       20125.56  50338.53 -46793.64     23670.45        -0.0
2025-06       23670.45  47137.79 -42361.59     28446.65         0.0
2025-07       28446.65  39162.28 -40682.07     26926.86        -0.0
2025-08       26926.86  43255.82 -43569.66     26613.02        -0.0
2025-10       26357.92  45276.40 -49895.60     21738.72        -0.0
2025-11       21738.72  45210.80 -47720.03     19229.49         0.0
2025-12       19229.49  58664.37 -57965.43     19928.43         0.0
2026-01       19928.43  43644.78 -40715.50     22857.71         0.0
2026-02       22857.71  40436.09 -38641.29     2

In [ ]:
#Conferencia de continuidade de mês.
#O saldo incial de um mês tem que ser igual ao saldo final do mês anterior.
#Teremos um erro entre agosto e outubro, pois não se tem dados de setmbro de 25 na base.
#A diferença do saldo desses dois meses vai ajudar a descobrir o saldo líquido do mês de setembro, que não se tem dados.

resumo["saldo_final_mes_anterior"] = resumo["saldo_final"].shift(1)
resumo["dif_na_virada"] = (resumo["saldo_inicial"] - resumo["saldo_final_mes_anterior"]).round(2)

print(resumo[["saldo_final_mes_anterior", "saldo_inicial", "saldo_final", "dif_na_virada"]].to_string())

#o output mostra que todo saldo incial é igual ao saldo final do mês anterior, e caso tivesse diferenças mostraria tambem na coluna  di na virada.
#A coluna nos mostra o saldo liquido do mês faltante.


         saldo_final_mes_anterior  saldo_inicial  saldo_final  dif_na_virada
mês                                                                         
2025-01                       NaN       19840.55     11140.63            NaN
2025-02                  11140.63       11140.63     50216.20            0.0
2025-03                  50216.20       50216.20     10842.18            0.0
2025-04                  10842.18       10842.18     20125.56            0.0
2025-05                  20125.56       20125.56     23670.45           -0.0
2025-06                  23670.45       23670.45     28446.65            0.0
2025-07                  28446.65       28446.65     26926.86            0.0
2025-08                  26926.86       26926.86     26613.02            0.0
2025-10                  26613.02       26357.92     21738.72         -255.1
2025-11                  21738.72       21738.72     19229.49           -0.0
2025-12                  19229.49       19229.49     19928.43            0.0